# 21.3 Spark MLlib:分布式机器学习 / Spark MLlib: Distributed Machine Learning

**中文**:当训练数据大到**单机内存装不下**(几百 GB 到 TB),连 scikit-learn 都无能为力——数据根本读不进来。**Spark MLlib** 让机器学习**跑在整个集群上**:数据分散在很多机器,训练也分散进行。但一个关键问题是:*机器学习算法怎么可能"分布式"?梯度下降不是要看所有数据吗?* 本节揭示答案,而且是最优雅的那种:**很多算法可以写成"每个分区各算一部分,再把结果加起来"的形式**——梯度就是可加的!我们**从零实现一个分布式逻辑回归**,证明它和单机训练**数学上完全等价**(权重差异约 $10^{-16}$),让你彻底理解 MLlib 底层在干什么,而不只是背 API。
**English**: When training data is too big to fit in **single-machine memory** (hundreds of GB to TB), even scikit-learn is helpless — the data won't even load. **Spark MLlib** runs machine learning **across an entire cluster**: data spread over many machines, training distributed too. But a key question: *how can an ML algorithm possibly be "distributed"? Doesn't gradient descent need to see all the data?* This section reveals the answer, in its most elegant form: **many algorithms can be written as "each partition computes a piece, then sum the results"** — gradients are additive! We **build a distributed logistic regression from scratch** and prove it is **mathematically identical to single-machine training** (weight difference ~$10^{-16}$), so you truly understand what MLlib does underneath rather than just memorizing the API.

---

**中文**:**数据并行(data parallelism)是分布式训练的核心**。关键数学事实:**很多损失函数的梯度是"每个样本梯度的和"**:
**English**: **Data parallelism is the core of distributed training.** The key mathematical fact: **the gradient of many loss functions is "a sum of per-sample gradients"**:
$$\nabla L(w)=\sum_{i=1}^{N}\nabla \ell_i(w)=\underbrace{\sum_{i\in P_1}\nabla \ell_i(w)}_{\text{分区1 局部算}}+\underbrace{\sum_{i\in P_2}\nabla \ell_i(w)}_{\text{分区2 局部算}}+\cdots$$

**中文**:因为求和可以拆分,分布式训练每一轮就变成三步:
**English**: Because summation can be split, each round of distributed training becomes three steps:
1. **中文**:**Map(局部计算)**:每个执行器在**自己的分区**上算局部梯度和(只看自己那部分数据)。
   **Map (local compute)**: each executor computes a partial gradient sum on **its own partition** (seeing only its slice of data).
2. **中文**:**Reduce / treeAggregate(聚合)**:Driver 把所有分区的局部梯度**加起来**得到全局梯度(用树形聚合减少通信瓶颈)。
   **Reduce / treeAggregate (aggregate)**: the Driver **sums** all partitions' partial gradients into the global gradient (tree-shaped aggregation to reduce the communication bottleneck).
3. **中文**:**Broadcast(广播)**:Driver 更新权重后,把**新权重广播**回所有执行器,进入下一轮。
   **Broadcast**: after the Driver updates the weights, it **broadcasts the new weights** back to all executors for the next round.

**中文**:**只要一个算法能写成"局部计算 + 求和聚合",它就能分布式**。线性/逻辑回归(梯度可加)、树模型(直方图可加)、K-means(簇统计可加)、朴素贝叶斯(计数可加)都天然适合。**MLlib 的 Pipeline API**(`Transformer`/`Estimator`/`Pipeline`)则和 sklearn 几乎一样,只是底层跑在分布式数据上。
**English**: **As long as an algorithm can be written as "local compute + sum aggregation," it can be distributed.** Linear/logistic regression (additive gradients), tree models (additive histograms), K-means (additive cluster stats), naive Bayes (additive counts) are all naturally suited. **MLlib's Pipeline API** (`Transformer`/`Estimator`/`Pipeline`) is almost identical to sklearn's, just running on distributed data underneath.

> 💡 **面试速查 / Interview cheat-sheet（★★ 分布式 ML 必考）**
> **中文**:**分布式 ML 核心=数据并行**:梯度是各样本梯度之和→可拆到各分区并行算, 再聚合(**Map 局部梯度 → treeAggregate 求和 → Broadcast 新权重**), 数学上与单机等价。**能分布式的算法**:线性/逻辑回归(梯度可加)、GBDT/随机森林(直方图统计可加, 如分布式 XGBoost)、K-means(簇心统计可加)、朴素贝叶斯(计数可加)。**难分布式**:强序列依赖的(如原始的逐样本 SGD)、需要全局排序的。**MLlib API**:`VectorAssembler`(拼特征成向量)、`StringIndexer`/`OneHotEncoder`、`Pipeline`(Transformer+Estimator 串联)、`CrossValidator`(分布式调参)。**Transformer**(有 `transform`, 如 scaler/已训练模型)vs **Estimator**(有 `fit`, 返回 Transformer)。**通信是瓶颈**:每轮要广播权重+聚合梯度→参数越多越贵→大模型用参数服务器/AllReduce(见 21.12)。**何时用**:数据 >单机内存才用 MLlib; 能进单机就用 sklearn/XGBoost(**快得多**)。**vs 分布式深度学习**:MLlib 主打经典 ML(树/线性), 深度学习用 Horovod/PyTorch DDP。面试金句:*"分布式训练靠数据并行——梯度可加, 每个分区算局部梯度、driver 树形聚合求和、再广播新权重, 与单机数学等价; 线性模型/树/K-means 都能这么并行; 通信(广播+聚合)是瓶颈, 且只有数据超单机内存才值得用 MLlib。"*
> **English**: **Distributed ML core = data parallelism**: the gradient is a sum of per-sample gradients → split across partitions to compute in parallel, then aggregate (**Map partial gradients → treeAggregate sum → Broadcast new weights**), mathematically equivalent to single-machine. **Distributable algorithms**: linear/logistic regression (additive gradients), GBDT/random forest (additive histogram stats, e.g. distributed XGBoost), K-means (additive centroid stats), naive Bayes (additive counts). **Hard to distribute**: strongly sequential ones (e.g. raw per-sample SGD), those needing global sorts. **MLlib API**: `VectorAssembler` (assemble features into a vector), `StringIndexer`/`OneHotEncoder`, `Pipeline` (chain Transformers + Estimators), `CrossValidator` (distributed tuning). **Transformer** (has `transform`, e.g. a scaler/trained model) vs **Estimator** (has `fit`, returns a Transformer). **Communication is the bottleneck**: each round broadcasts weights + aggregates gradients → costlier with more parameters → big models use parameter servers / AllReduce (see 21.12). **When to use**: only when data > single-machine memory; if it fits, use sklearn/XGBoost (**much faster**). **vs distributed deep learning**: MLlib targets classic ML (trees/linear), deep learning uses Horovod/PyTorch DDP. Interview line: *"Distributed training relies on data parallelism — gradients are additive, so each partition computes a partial gradient, the driver tree-aggregates the sum, then broadcasts new weights, mathematically equivalent to single-machine; linear models/trees/K-means all parallelize this way; communication (broadcast + aggregate) is the bottleneck, and MLlib is worth it only when data exceeds single-machine memory."*


In [ ]:

# ============================================================
# 从零实现分布式逻辑回归 / distributed logistic regression from scratch
# 中文:造 4 万样本(假装"单机装不下"), 分到 8 个"执行器"。每轮:各分区算局部梯度→driver 求和→更新→广播。
#      对比单机逐样本训练——证明两者数学上完全等价(权重差 ~1e-16)。
# English: 40k samples (pretend "won't fit on one machine"), split over 8 "executors". Each round: partitions compute
#      partial gradients → driver sums → update → broadcast. Compare to single-machine — prove exact equivalence.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
N,D=40000,15
X=np.random.randn(N,D); w_true=np.random.randn(D); y=(X@w_true+0.5*np.random.randn(N)>0).astype(float)
sigmoid=lambda z: 1/(1+np.exp(-z))

def train_single(epochs=50, lr=0.3):                 # 单机基线(能看到所有数据)/ single-machine baseline
    w=np.zeros(D); curve=[]
    for _ in range(epochs):
        g=X.T@(sigmoid(X@w)-y)/N                      # 全量梯度 / full-batch gradient
        w-=lr*g; curve.append(((sigmoid(X@w)>0.5)==y).mean())
    return w, curve

def train_distributed(nparts=8, epochs=50, lr=0.3):  # 分布式(数据分区, 只聚合梯度)/ distributed
    parts=[(X[i], y[i]) for i in np.array_split(np.arange(N), nparts)]   # 数据分到 8 个执行器 / split to 8 executors
    w=np.zeros(D); curve=[]
    for _ in range(epochs):
        # --- Map:每个执行器在自己分区上算 (局部梯度和, 样本数), 只看自己的数据 ---
        partials=[(Xp.T@(sigmoid(Xp@w)-yp), len(yp)) for Xp,yp in parts]
        # --- Reduce / treeAggregate:driver 把各分区局部梯度加起来 ---
        gsum=sum(p[0] for p in partials); n=sum(p[1] for p in partials)
        w-=lr*(gsum/n)                                # --- driver 更新权重, 下一轮 Broadcast 回执行器 ---
        curve.append(((sigmoid(X@w)>0.5)==y).mean())
    return w, curve

ws,cs=train_single(); wd,cd=train_distributed()
print(f"单机 vs 分布式 权重最大差异 / max weight diff: {np.abs(ws-wd).max():.2e}  (≈机器精度→完全等价!)")
print(f"单机最终训练准确率 / single-machine final acc:  {cs[-1]:.4f}")
print(f"分布式最终训练准确率 / distributed final acc:    {cd[-1]:.4f}")
print("→ 把梯度按分区拆开算再求和, 与单机逐样本梯度下降结果一模一样。这就是 data parallelism。")


In [ ]:

# ============================================================
# MLlib 的 Pipeline API 思想:Transformer / Estimator / Pipeline / mini pipeline abstraction
# 中文:MLlib(和 sklearn)用统一抽象:Transformer(有 transform, 如标准化/已训练模型)、
#      Estimator(有 fit, 返回 Transformer)、Pipeline(把它们串起来 fit/transform)。从零复现这个模式。
# English: MLlib (like sklearn) uses a unified abstraction: Transformer (has transform), Estimator (has fit, returns
#      a Transformer), Pipeline (chains them). Reproduce the pattern from scratch.
# ============================================================
class StandardScaler:                                 # Estimator→Transformer / 标准化
    def fit(self,X): self.mu=X.mean(0); self.sd=X.std(0)+1e-9; return self   # fit 学统计量 / learn stats
    def transform(self,X): return (X-self.mu)/self.sd                        # transform 应用 / apply
class LogRegEstimator:                                 # Estimator(fit 返回训练好的 Transformer 模型)
    def fit(self,X,y):
        w=np.zeros(X.shape[1])
        for _ in range(60): w-=0.3*X.T@(sigmoid(X@w)-y)/len(y)               # 训练(分布式同理)/ train
        m=LogRegModel(); m.w=w; return m
class LogRegModel:                                     # Transformer(已训练模型, 有 transform=预测)
    def transform(self,X): return (sigmoid(X@self.w)>0.5).astype(float)
class Pipeline:                                        # 串联 stages, 统一 fit/transform / chain stages
    def __init__(self,stages): self.stages=stages
    def fit(self,X,y):
        self.fitted=[]
        for s in self.stages:
            if hasattr(s,"fit") and isinstance(s,LogRegEstimator): s=s.fit(X,y)   # Estimator 用 (X,y) fit
            elif hasattr(s,"fit"): s=s.fit(X); X=s.transform(X)                    # 预处理 Estimator fit+transform
            self.fitted.append(s)
        return self
    def transform(self,X):
        for s in self.fitted: X=s.transform(X)
        return X
pipe=Pipeline([StandardScaler(), LogRegEstimator()]).fit(X,y)   # 标准化 → 逻辑回归, 一条流水线 / one pipeline
acc=(pipe.transform(X)==y).mean()
print(f"Pipeline(标准化→逻辑回归)训练准确率 / pipeline train acc: {acc:.4f}")
print("这正是 MLlib/sklearn 的 Pipeline 模式:Estimator.fit→Transformer, 多阶段串联, 端到端 fit/transform")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 收敛曲线:分布式与单机重合 / convergence curves overlap
ax[0].plot(cs,"k-",lw=3,label="单机 single-machine",alpha=0.5)
ax[0].plot(cd,"o-",color="#C44E52",ms=3,label="分布式 distributed (8 partitions)")
ax[0].set_xlabel("训练轮次 epoch"); ax[0].set_ylabel("训练准确率"); ax[0].legend()
ax[0].set_title("分布式训练曲线与单机完全重合(数学等价)")
# ② 数据并行示意 / data-parallel diagram
ax[1].axis("off"); ax[1].set_title("数据并行:Map 局部梯度 → 聚合 → 广播",fontsize=12,weight="bold")
for i,x0 in enumerate([0.05,0.28,0.51,0.74]):
    ax[1].add_patch(plt.Rectangle((x0,0.55),0.19,0.16,fc="#55A868",alpha=0.6,transform=ax[1].transAxes))
    ax[1].text(x0+0.095,0.63,f"分区{i+1}\n局部梯度",ha="center",va="center",fontsize=8,transform=ax[1].transAxes)
    ax[1].annotate("",xy=(0.5,0.42),xytext=(x0+0.095,0.55),arrowprops=dict(arrowstyle="->",color="gray"),transform=ax[1].transAxes)
ax[1].add_patch(plt.Rectangle((0.32,0.28),0.36,0.13,fc="#4C72B0",alpha=0.7,transform=ax[1].transAxes))
ax[1].text(0.5,0.345,"Driver: Σ 局部梯度 → 更新权重",ha="center",va="center",color="white",fontsize=9,transform=ax[1].transAxes)
ax[1].annotate("广播新权重 broadcast w",xy=(0.15,0.55),xytext=(0.32,0.28),
               arrowprops=dict(arrowstyle="->",color="#DD8452"),fontsize=8,transform=ax[1].transAxes)
ax[1].text(0.5,0.12,"梯度可加 → 每轮拆开算再求和 → 与单机等价\ngradients are additive → split & sum each round → same as single-machine",
           ha="center",fontsize=8.5,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big03_viz.png",dpi=80); plt.show()
print("左:两条曲线完全重合; 右:每轮各分区算局部梯度→driver 聚合求和→广播新权重")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **分布式 ML 不是"另一种算法",而是同一算法的可加分解**:很多人以为分布式训练是某种玄学近似,其实对逻辑回归这类模型,它和单机**逐比特相同**(权重差 $10^{-16}$ 是浮点求和顺序造成的,连近似都算不上)。原因就一句话:**梯度是每个样本梯度的和,而和可以拆开在不同机器上算再加回来**。理解了这一点,你就理解了 MLlib、分布式 XGBoost、乃至深度学习数据并行(21.12)的共同底层——它们都是"局部算 + 聚合 + 广播"这个循环。
2. **Pipeline 抽象让分布式和单机代码几乎一样**:MLlib 刻意模仿 sklearn 的 `Transformer`/`Estimator`/`Pipeline` 抽象,所以你写分布式 ML 的**代码结构**和写 sklearn 几乎没区别——`VectorAssembler` 拼特征、`StringIndexer` 编码、`Pipeline` 串联、`CrossValidator` 调参。真正的分布式细节(数据分区、梯度聚合、广播)被引擎藏起来了。这是好事(易上手),但也是坑:**它让人误以为"分布式很简单",从而在不需要的时候滥用**。
3. **诚实的边界:MLlib 常常是错误的选择**。①**通信开销**:每一轮迭代都要广播权重、聚合梯度,这在网络上来回。对**迭代很多**的算法(逻辑回归几百轮),通信可能比计算还贵——所以 MLlib 在中等数据上经常**比单机 sklearn/XGBoost 慢好几倍**。②**不是所有算法都并行得好**:线性模型、树、K-means 可加所以很好;但强序列依赖的算法(某些在线学习)、需要全局排序或全局最近邻的算法,分布式代价极高。③**最重要的判断**:只有当数据**真的超过单机内存**(几百 GB+)时,MLlib 才值得。现实中绝大多数"大数据 ML"其实几个 GB,**用 XGBoost/LightGBM 单机版又快又准又简单**,上 Spark MLlib 纯属自找麻烦。**先算清楚数据到底多大,再决定要不要分布式——这比会调 MLlib 的 API 重要得多。**

**English**:
1. **Distributed ML isn't "another algorithm" but the same algorithm's additive decomposition**: many think distributed training is some mystical approximation, but for models like logistic regression it is **bit-for-bit identical** to single-machine (the $10^{-16}$ weight difference is just floating-point summation order, not even an approximation). The reason is one sentence: **the gradient is a sum of per-sample gradients, and a sum can be split across machines and added back**. Grasp this and you understand the common foundation of MLlib, distributed XGBoost, and even deep-learning data parallelism (21.12) — all the same "local compute + aggregate + broadcast" loop.
2. **The Pipeline abstraction makes distributed and single-machine code nearly identical**: MLlib deliberately mimics sklearn's `Transformer`/`Estimator`/`Pipeline`, so the **code structure** of distributed ML looks almost the same as sklearn — `VectorAssembler` to assemble features, `StringIndexer` to encode, `Pipeline` to chain, `CrossValidator` to tune. The real distributed details (partitioning, gradient aggregation, broadcast) are hidden by the engine. This is good (easy to start) but also a trap: **it makes people think "distributed is easy" and overuse it when unneeded**.
3. **Honest limits: MLlib is often the wrong choice**. ① **Communication overhead**: every iteration broadcasts weights and aggregates gradients over the network. For **many-iteration** algorithms (logistic regression's hundreds of rounds), communication can cost more than computation — so on medium data MLlib is often **several times slower than single-machine sklearn/XGBoost**. ② **Not all algorithms parallelize well**: linear models, trees, K-means are additive and fine; but strongly sequential algorithms (some online learning), or those needing global sorts / global nearest neighbors, are very costly to distribute. ③ **The most important judgment**: MLlib is worth it only when data **truly exceeds single-machine memory** (hundreds of GB+). In reality most "big-data ML" is actually a few GB, where **single-machine XGBoost/LightGBM is faster, more accurate, and simpler** — reaching for Spark MLlib just invites trouble. **Figure out how big the data really is before deciding to distribute — that matters far more than knowing MLlib's API.**

> 💼 **实战视角 / Practical angle**
> **中文**:分布式 ML 落地:①**先确认数据规模**——单机内存(现在云主机常有几百 GB RAM)放得下就用 **XGBoost/LightGBM/sklearn**(几乎总是更快);真超内存才上 MLlib。②MLlib 用法和 sklearn 像:`VectorAssembler`→`StringIndexer`→模型→`Pipeline`→`CrossValidator`;树模型用 `GBTClassifier`。③**减少迭代通信**:能用一次性统计的算法(树、朴素贝叶斯)比多轮迭代的(逻辑回归)在分布式下更划算。④**大规模树模型**用分布式 XGBoost/LightGBM(直方图可加, 通信少)常优于 MLlib。⑤**深度学习**别用 MLlib——用 PyTorch DDP/Horovod(21.12)。⑥缓存训练数据 `df.cache()`(迭代算法反复读)。面试金句:*"分布式训练=数据并行:梯度可加, 各分区算局部梯度、树形聚合求和、广播新权重, 与单机等价; MLlib 的 Pipeline 抽象和 sklearn 一样; 但通信开销大, 只有数据超单机内存才值得——多数场景 XGBoost/LightGBM 单机更快, 别为几 GB 数据上 Spark。"*
> **English**: Distributed ML in practice: ① **confirm data size first** — if it fits single-machine memory (cloud machines now often have hundreds of GB RAM), use **XGBoost/LightGBM/sklearn** (almost always faster); only truly-over-memory data warrants MLlib. ② MLlib usage resembles sklearn: `VectorAssembler` → `StringIndexer` → model → `Pipeline` → `CrossValidator`; trees via `GBTClassifier`. ③ **Minimize iterative communication**: algorithms using one-pass statistics (trees, naive Bayes) beat many-round iterative ones (logistic regression) when distributed. ④ **Large-scale tree models** use distributed XGBoost/LightGBM (additive histograms, less communication), often beating MLlib. ⑤ **Deep learning** — don't use MLlib; use PyTorch DDP/Horovod (21.12). ⑥ Cache training data `df.cache()` (iterative algorithms re-read repeatedly). Interview line: *"Distributed training = data parallelism: gradients are additive, so partitions compute partial gradients, tree-aggregate the sum, and broadcast new weights, equivalent to single-machine; MLlib's Pipeline abstraction mirrors sklearn; but communication overhead is high, so it's worth it only when data exceeds single-machine memory — in most cases single-machine XGBoost/LightGBM is faster; don't use Spark for a few GB."*

---
### 小结 / Summary
- **中文**:分布式 ML=数据并行:梯度可加 → 各分区算局部梯度 → treeAggregate 求和 → 广播权重, 与单机数学等价。
- **English**: Distributed ML = data parallelism: additive gradients → partitions compute partial gradients → treeAggregate sum → broadcast weights, mathematically equivalent to single-machine.
- **中文**:MLlib 的 Transformer/Estimator/Pipeline 抽象与 sklearn 一致; 线性/树/K-means 天然可并行。
- **English**: MLlib's Transformer/Estimator/Pipeline mirrors sklearn; linear/tree/K-means parallelize naturally.
- **中文**:通信开销大, 只有数据超单机内存才用 MLlib; 多数场景 XGBoost/LightGBM 单机更快更简单。
- **English**: High communication overhead; use MLlib only when data exceeds single-machine memory; usually single-machine XGBoost/LightGBM is faster and simpler.
